# 引入格点偏移的 RKS 一阶梯度

在讨论二阶梯度格点权重问题前，我们先熟悉一阶梯度格点权重导数的实现。

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pyscf import gto, dft, lib, grad, hessian, data
from pyscf.hessian import thermo
import numpy as np
from functools import partial
import scipy

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [3]:
import sys
sys.path.append("..")

from pyhessref.nimatmul.becke_partition import becke_partition
from pyhessref.nimatmul import rks as rks_nimatmul

我们使用下述虚假的氢键结合的 H2O-HF 体系作为例子。但出于数值分析的目的，尽管该体系是平面体系，可以确定主轴以比较清楚地定义分子坐标，但我们实际上对齐作旋转，把一些潜在的数值问题凸显出来。

In [4]:
# xyz = """
# O     0.00000000     0.00000000     0.12982363
# H     0.75933475     0.00000000    -0.46621158
# H    -0.75933475     0.00000000    -0.46621158
# F     0.00000000     0.00000000     1.80000000
# H     0.00000000     0.00000000     1.00000000
# """
xyz = """
O   0.08408549239940    0.09520488520532   -0.02682973550689
H  -0.03819269785235   -0.37439900848671    0.80765661029833
H  -0.56572864168501   -0.30938436677850   -0.61495928113491
F   1.16584235334441    1.32001233804336   -0.37199332596388
H   0.64769019630245    0.73334018780187   -0.20666295886882
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

## 梯度正确性验证策略

我们将使用 B3LYP 作为例子。

In [5]:
mf_b3lyp = dft.RKS(mol, xc="B3LYP").run()
mf_b3lyp_grad = mf_b3lyp.Gradients().run()
mf_b3lyp_grad_resp = mf_b3lyp.Gradients().run(grid_response=True)

converged SCF energy = -176.702837293021
--------------- RKS gradients ---------------
         x                y                z
0 O     0.3821684412     0.4327085060    -0.1219381814
1 H    -0.0010372365     0.0017122590    -0.0066018436
2 H     0.0035607879     0.0011455899     0.0057964983
3 F    -0.3746607338    -0.4242052676     0.1195350531
4 H    -0.0100337345    -0.0113607481     0.0032019051
----------------------------------------------
--------------- RKS gradients ---------------
         x                y                z
0 O     0.3821695134     0.4327071609    -0.1219398286
1 H    -0.0010381699     0.0017114536    -0.0066021903
2 H     0.0035596025     0.0011429506     0.0057971307
3 F    -0.3746562014    -0.4242012072     0.1195410409
4 H    -0.0100347446    -0.0113603578     0.0032038472
----------------------------------------------


对于当前的例子，我们会注意到引入格点偏移所导致的误差是非常小的，只有 1e-6 级别。因此，我们必须要非常谨慎地对待这一项的计算。

我们之所以要考虑格点权重，有两方面因素：
- 一部分特殊的计算，对分子力有严格的要求，否则可能会导致动力学轨迹偏移；
- 二阶梯度的格点偏移误差会被严重放大，为此我们需要首先确认一阶梯度的数值。

In [6]:
de_grids = mf_b3lyp_grad.de - mf_b3lyp_grad_resp.de
np.abs(de_grids).max()

np.float64(5.987875170809787e-06)

1e-6 级别的数值误差，不管是数值-解析梯度相互验证、或者是不同软件之间的数值对比，都是可以接受的。

但为了真正地、有效地实现格点偏移导数，我们需要其他的数值验证手段，即 **能量对原子坐标的平动不变性**。

具体来说，记能量是原子坐标的函数 $E(\bm{A}, \bm{B}, \cdots)$，则对于 $t$ 分量 ($t \in \{x, y, z\}$，当然它也可以是任意单位向量的方向)，引入微小量 $\bm{\epsilon}$ 是 $t$ 分量方向向量，我们有

$$
\frac{\mathrm{d} E (\bm{A} + \bm{\epsilon}, \bm{B} + \bm{\epsilon}, \cdots)}{\mathrm{d} \bm{\epsilon}} = \frac{\partial E}{\partial \bm{A}} + \frac{\partial E}{\partial \bm{B}} + \cdots = 0
$$

现在我们取 $t \in \{x, y, z\}$，那么意味着

$$
\sum_{M} \frac{\partial E}{\partial M_t} = 0
$$

但实际程序总会有数值误差。对于没有引入格点权重的 RKS 一阶梯度，数值误差大约在 1e-6 级别；而对于引入格点权重的 RKS 一阶梯度，数值误差大约在 1e-14 级别。

In [7]:
(np.abs(mf_b3lyp_grad.de.sum(axis=0)).max(), np.abs(mf_b3lyp_grad_resp.de.sum(axis=0)).max())

(np.float64(6.568574708487596e-06), np.float64(1.2656542480726785e-14))

对于 RKS 一阶梯度，我们要求 $\sum_{M} \frac{\partial E}{\partial M_t} = 0$ 应该要在 1e-12 级别。

## 计算格点偏移梯度

以前我们总是认为，格点权重及其坐标是不动的。

$$
E^\text{xc} = \sum_g w_g f_g(\bm{R})
$$

但实际上，格点权重也与原子坐标有关，泛函格点也是在具体的格点坐标下产生的：

$$
E^\text{xc} = \sum_g w_g(\bm{R}) f_g(\bm{R}, \bm{r}_g (\bm{R}))
$$

格点偏移梯度要分为两部分：

- 格点权重梯度 (全导数)

    $$
    \frac{\mathrm{d} E^\text{xc}}{\mathrm{d} \bm{R}} \leftarrow \sum_g \frac{\mathrm{d} w_g}{\mathrm{d} \bm{R}} f_g
    $$

- 泛函对格点偏移梯度 (隐偏导数)

    $$
    \frac{\mathrm{d} E^\text{xc}}{\mathrm{d} \bm{R}} \leftarrow \sum_g w_g \frac{\partial f_g}{\partial \bm{r}_g} \frac{\mathrm{d} \bm{r}_g}{\mathrm{d} \bm{R}}
    $$

这两部分数值通常相反，加起来接近零，但如前述，这个数值在 1e-6 级别，是格点偏移导致的梯度。

In [8]:
grids = mf_b3lyp.grids
ngrids = grids.weights.size
ni = mf_b3lyp._numint
ao = ni.eval_ao(mol, grids.coords, deriv=2)
dm0 = mf_b3lyp.make_rdm1()
ao_dm0 = ao @ dm0
rho, exc, vxc, fxc = rks_nimatmul._eval_rho_exc_vxc_fxc("B3LYP", "GGA", ao, ao_dm0)
drho = rks_nimatmul._make_drho("GGA", ao, ao_dm0, mol.aoslice_by_atom())

In [9]:
natm = mol.natm
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
adjustment_factor = np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
becke_result = becke_partition(grids.coords, mol.atom_coords(), grids.atm_idx, grids.quadrature_weights, adjustment_factor, 3, 512, 2, None)
w, dw, ddw = becke_result["w"], becke_result["dw"], becke_result["ddw"]

### 格点权重梯度

格点权重梯度，在有了 Becke partition 导数之后，是非常容易拿到的。当然，格点权重 partition 不止有 Becke scheme，但作为梯度实现的人所接触的 API 而言，应该得是大同小异的。

当然，要求高一些的话，我们应该不需要输出 `(natm, 3, ngrids)` 的格点权重导数，而是直接在计算 Becke partition 导数的时候，融合算子作 contraction，直接输出 `(natm, 3)` 的格点权重导数。这在 Rust 代码里已经实现了。

In [10]:
t1 = np.einsum("g, g, Atg -> At", exc, rho[0], dw)

### 泛函对格点偏移梯度

格点坐标偏移的导数有两种计算方法。尽管 REST analdrv 或者 pyhessref 并不严格是这个流程，但总地来说，梯度计算有两种策略：

- 将原子导数计算先化归到基组问题，通过 $\phi_{t g \mu} \phi_{g \nu} \rightarrow V_{t \mu \nu}$ 矩阵，随后依 $\mu \in A$ 所在原子作缩并得到维度为 $(A, t)$ 的张量；
- 先得到密度格点导数 $\rho_{A_t g}$，与 $f_g^\rho$ 暴力缩并。

只从 FLOPs 计算量上来看，其实两者应该是接近的；因为密度格点导数 $\rho_{A_t g}$ 尽管是用密度矩阵算的，但每个 $A$ 只负责密度矩阵与该原子基组相关的部分，FLOPs 并不是增加一个复杂度的。但无论如何，我们仍然要输出一个 `(natm, 3, nvars, ngrids)` 的张量 (GGA/MGGA 实际上应该是 $\rho_{A_t \xi g}$ 的张量)，这会引入不算太小的 memory footprint。因此，DFT 通常在计算一阶梯度时，采用的是第一种做法。

在计算格点偏移梯度时，我们也可以考虑两种策略。这两种策略各有优缺点。首先我们要指出，

$$
\frac{\mathrm{d} E^\text{xc}}{\mathrm{d} A_t} \leftarrow \sum_g w_g \frac{\partial f_g}{\partial r_{t g}} \frac{\mathrm{d} r_{t g}}{\mathrm{d} A_t} \delta_{g \in A}
$$

我们着重讨论第一种，即化归到基组问题。

其中格点坐标是与原子坐标同步移动的，因此其导数为 1。第一种做法 (先化归到基组问题) 的情况大概用公式示意为 (不是严格推演)

$$
\begin{align*}
\frac{\mathrm{d} E^\text{xc}}{\mathrm{d} A_t} 
&\leftarrow \sum_g w_g \frac{\partial f_g}{\partial \rho_{g}} \frac{\mathrm{d} \rho_{g}}{\mathrm{d} r_g} \delta_{g \in A} \\
&= \sum_g w_g \frac{\partial f_g}{\partial \rho_{g}} \frac{\partial \xi_{\mu \nu g}}{\partial r_g} D_{\mu \nu} \delta_{g \in A}
\end{align*}
$$

在不引入格点偏移的 DFT 导数计算中，化归为基组问题的意思是先对格点求和 $\sum_g w_g \frac{\partial f_g}{\partial \rho_{g}} \frac{\partial \xi_{\mu \nu g}}{\partial r_g}$ 得到 $V_{t \mu \nu}$，随后对 $D_{\mu \nu} \delta_{\mu \in A}$ 求和。

但对于格点偏移 DFT 导数计算中，我们不再是 $\delta_{\mu \in A}$ 而是 $\delta_{g \in A}$；这意味着 $\delta$ 的作用要提前在缩并格点的时候就要完成，且被缩并的是完整的密度矩阵而不是依原子 slice 的。它将带来的 non-trivial 影响是

- 在处理 $V_{t \mu \nu}$ 或类似地 $\mathscr{T}_{t \mu}$ 时，我们需要同时 tag 当前格点分批是在哪个原子上。同时，**不同原子所产生的 Lebedev 格点，不能一起缩并**。
- 尽管不是必须，但最好**传入的格点需要按照原子进行分组，而不能是乱序的**。有时乱序 (重新排序) 会使得空间上接近的格点在相同的 batch 中 (但实际上是不同的原子的 Lebedev 产生的格点)；这种重新排序有时可以利用稀疏性加速计算，但在当前问题中反而会导致程序编写难度与 memory footprint 的增加。

只要满足这两个条件，格点偏移的导数是可以与普通 DFT 导数同时计算的。

In [11]:
t2 = np.zeros((natm, 3))
t3 = np.zeros((natm, 3))
nao = mol.nao

for A in range(natm):
    mask_A = grids.atm_idx == A
    w_A = w[mask_A]
    vxc_A = vxc[:, mask_A]
    ao_A = ao[:, mask_A, :]

    wv_A = w_A * vxc_A
    wv_A[0] *= 0.5
    vtmp = np.zeros((3, nao, nao))
    grad.rks._gga_grad_sum_(vtmp, mol, ao_A, wv_A, None, None)
    t2[A] += 2 * np.einsum("tuv, uv -> t", vtmp, dm0)

    rho_A = drho[:, :, :, mask_A]
    t3[A] -= np.einsum("g, xg, Atxg -> t", w_A, vxc_A, rho_A)
t2, t3

(array([[-0.03422, -0.03874,  0.01092],
        [-0.06401, -0.26574,  0.48464],
        [-0.37187, -0.2278 , -0.34556],
        [ 0.71787,  0.8128 , -0.22906],
        [-0.24776, -0.28053,  0.07906]]),
 array([[-0.03422, -0.03874,  0.01092],
        [-0.06401, -0.26574,  0.48464],
        [-0.37187, -0.2278 , -0.34556],
        [ 0.71787,  0.8128 , -0.22906],
        [-0.24776, -0.28053,  0.07906]]))

In [12]:
np.abs(t3 - t2).max()

np.float64(3.9968028886505635e-15)

最后我们可以验证，引入格点权重导数、以及泛函对格点偏移导数后，对原子求和的导数结果非常接近零。

In [13]:
np.abs((mf_b3lyp_grad.de + t1 + t2).sum(axis=0)).max()

np.float64(1.609823385706477e-14)